<a href="https://colab.research.google.com/github/ZYZY2727/2024-Orbital-6532/blob/master/Beam_shaping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
Gauss -> flat-top two-plano-asphere beam shaper, Galilean (non-crossing) form.
Reproduces / generalises Gao et al., J. Mod. Opt. 71, 219 (2024).

Geometry (all surfaces on a common axis, light travels +z):

    S1: plano (entrance of lens 1)      -- ray enters at normal incidence
    S2: asphere z1(r),  vertex at z = 0 -- glass -> air
        air gap D
    S3: asphere z2(R),  vertex at z = D -- air -> glass
    S4: plano (exit of lens 2)          -- ray exits at normal incidence

Both aspheres face each other. Lens 1 is plano-CONCAVE (negative), lens 2 is
plano-CONVEX (positive): that is the Galilean arrangement, no internal focus.

Design is EXACT (no ODE integration, no optimisation):
  1. ray map      R(r)  from energy conservation
  2. sag of S2    dz1/dr = dr / sqrt( D^2 (n-1)^2 + (n^2-1) dr^2 ),  dr = R - r
  3. sag of S3    z2 = z1 + s,  s = [-D(n-1) + sqrt(D^2(n-1)^2 + (n^2-1)dr^2)]/(n^2-1)
Both follow from constant optical path length + vector Snell at each asphere.
Derivations are in the accompanying notes.

The script then fits a ZEMAX "Even Asphere" to each exact sag and does an
independent real-ray trace through the FITTED surfaces to check the result.
"""

import numpy as np
from scipy.optimize import least_squares, brentq

# ----------------------------------------------------------------------------
# USER PARAMETERS  -- edit these
# ----------------------------------------------------------------------------
LAMBDA_UM = 0.638      # design wavelength [um]
W0        = 5.5        # input Gaussian 1/e^2 RADIUS [mm]   (paper: 11 mm dia)
R_TRUNC   = 5.5        # input clear semi-aperture [mm]      (= truncation radius)
N_GLASS   = 1.4568     # fused silica @ 638 nm
D_GAP     = 30.0       # vertex separation of the two aspheres [mm]
MAG       = 2.3227     # on-axis (paraxial) magnification dR/dr|_0
Q_ORDER   = np.inf     # flat-top edge order (np.inf = ideal top hat; 30-50 = soft)
CT1       = 5.0        # centre thickness lens 1 [mm]
CT2       = 5.0        # centre thickness lens 2 [mm]

NPTS      = 4001       # sampling of the ray map
N_ASPH    = 8          # number of even terms fitted: a4, a6, ... a14
APER_PAD  = 1.06       # clear-aperture margin on the fitted surfaces

# R_FL follows from the requested on-axis magnification:
#   R(r) -> R_FL * sqrt(2) * r / W0   as r -> 0
R_FL = MAG * W0 / np.sqrt(2.0)


# ----------------------------------------------------------------------------
# 1. RAY MAP
# ----------------------------------------------------------------------------
def ray_map(r, w0=W0, r_fl=R_FL, q=Q_ORDER):
    """Output ray height R for an input ray height r (energy conservation).

    Ideal top hat (q -> inf):      R = R_FL sqrt(u)
    Homogenised Lorentz order q:   R = R_FL sqrt(u) / (1 - u^(q/2))^(1/q)
    with u = 1 - exp(-2 r^2 / w0^2)  (encircled-energy fraction of the Gaussian).

    NOTE: eq. (6) of the paper prints the denominator as a square root; the
    correct exponent is 1/q, which is what reduces to their eq. (8) as q -> inf.
    """
    u = 1.0 - np.exp(-2.0 * (r / w0) ** 2)
    if not np.isfinite(q):
        return r_fl * np.sqrt(u)
    return r_fl * np.sqrt(u) / (1.0 - u ** (q / 2.0)) ** (1.0 / q)


# ----------------------------------------------------------------------------
# 2. EXACT SAGS
# ----------------------------------------------------------------------------
def exact_sags(r, n=N_GLASS, D=D_GAP):
    """Return R(r), z1(r), z2 evaluated at R(r)."""
    R = ray_map(r)
    dr = R - r
    A = D * (n - 1.0)
    B = n * n - 1.0
    root = np.sqrt(A * A + B * dr * dr)

    dz1dr = dr / root
    z1 = np.concatenate(([0.0], np.cumsum(0.5 * (dz1dr[1:] + dz1dr[:-1]) * np.diff(r))))

    s = (-A + root) / B          # z2 - z1, closed form from the OPL condition
    z2 = z1 + s
    return R, z1, z2


# ----------------------------------------------------------------------------
# 3. EVEN-ASPHERE FIT
# ----------------------------------------------------------------------------
def even_asphere(rho, c, k, a):
    """ZEMAX Even Asphere sag. a = [a4, a6, a8, ...] (coefficients of r^4, r^6...)."""
    arg = 1.0 - (1.0 + k) * c * c * rho * rho
    arg = np.where(arg < 0.0, np.nan, arg)
    z = c * rho * rho / (1.0 + np.sqrt(arg))
    for i, ai in enumerate(a):
        z = z + ai * rho ** (2 * i + 4)
    return z


def fit_asphere(rho, sag, c_fixed, n_terms=N_ASPH):
    """Fit k and a4..a(2n+2) with the vertex curvature held at its exact value."""
    scale = rho.max()

    def resid(p):
        k = p[0]
        a = p[1:] / scale ** (2 * np.arange(2, n_terms + 2))
        model = even_asphere(rho, c_fixed, k, a)
        out = model - sag
        return np.where(np.isfinite(out), out, 1e3)

    p0 = np.zeros(1 + n_terms)
    sol = least_squares(resid, p0, xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=20000)
    k = sol.x[0]
    a = sol.x[1:] / scale ** (2 * np.arange(2, n_terms + 2))
    rms = np.sqrt(np.nanmean((even_asphere(rho, c_fixed, k, a) - sag) ** 2))
    return k, a, rms


# ----------------------------------------------------------------------------
# 4. INDEPENDENT REAL-RAY TRACE THROUGH THE FITTED SURFACES
# ----------------------------------------------------------------------------
def sag_and_slope(rho, c, k, a):
    h = 1e-7 * max(1.0, abs(rho))
    z = even_asphere(np.array([rho]), c, k, a)[0]
    zp = (even_asphere(np.array([rho + h]), c, k, a)[0]
          - even_asphere(np.array([rho - h]), c, k, a)[0]) / (2 * h)
    return z, zp


def refract(d, normal, n1, n2):
    """2D vector Snell. `normal` need not point any particular way."""
    N = normal / np.linalg.norm(normal)
    if np.dot(d, N) > 0:
        N = -N                      # make N point back toward the incoming ray
    mu = n1 / n2
    ci = -np.dot(d, N)
    st2 = mu * mu * (1.0 - ci * ci)
    if st2 > 1.0:
        return None                 # TIR
    return mu * d + (mu * ci - np.sqrt(1.0 - st2)) * N


def trace(r_in, c1, k1, a1, c2, k2, a2, n=N_GLASS, D=D_GAP):
    """Ray starts inside lens 1, parallel to axis, at height r_in."""
    z1, z1p = sag_and_slope(r_in, c1, k1, a1)
    P = np.array([r_in, z1])
    d = refract(np.array([0.0, 1.0]), np.array([-z1p, 1.0]), n, 1.0)
    if d is None:
        return None

    def f(t):
        rr = P[0] + t * d[0]
        zz = P[1] + t * d[1]
        return zz - (D + even_asphere(np.array([rr]), c2, k2, a2)[0])

    t0 = (D - P[1]) / d[1]                 # first guess: flat plane at z = D
    lo, hi = 0.5 * t0, 1.5 * t0
    flo = f(lo)
    while not np.isfinite(f(hi)) or f(hi) * flo > 0:
        hi = lo + 0.7 * (hi - lo)
        if hi - lo < 1e-12:
            return None
    t = brentq(f, lo, hi, xtol=1e-13)
    Q = P + t * d
    _, z2p = sag_and_slope(Q[0], c2, k2, a2)
    d2 = refract(d, np.array([-z2p, 1.0]), 1.0, n)
    return Q[0], d2


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    n = N_GLASS
    r = np.linspace(0.0, R_TRUNC, NPTS)
    R, z1, z2 = exact_sags(r)

    c1 = (MAG - 1.0) / (D_GAP * (n - 1.0))      # exact vertex curvature, S2
    c2 = c1 / MAG                               # exact vertex curvature, S3

    Rmax = R[-1]
    r_fit = np.linspace(0.0, R_TRUNC * APER_PAD, NPTS)
    R_fit, z1_fit, z2_fit = exact_sags(r_fit)

    k1, a1, rms1 = fit_asphere(r_fit, z1_fit, c1)
    k2, a2, rms2 = fit_asphere(R_fit, z2_fit, c2)

    print("=" * 74)
    print("DESIGN INPUTS")
    print("=" * 74)
    print(f"  wavelength                 {LAMBDA_UM*1000:.1f} nm")
    print(f"  input Gaussian 1/e^2 waist w0     = {W0:.4f} mm  (dia {2*W0:.3f} mm)")
    print(f"  input truncation radius    r_max  = {R_TRUNC:.4f} mm")
    print(f"  truncation ratio           r_max/w0 = {R_TRUNC/W0:.3f}")
    print(f"  power inside r_max         = {100*(1-np.exp(-2*(R_TRUNC/W0)**2)):.2f} %")
    print(f"  glass index                n      = {n:.5f}")
    print(f"  asphere vertex separation  D      = {D_GAP:.3f} mm")
    print(f"  on-axis magnification      m0     = {MAG:.4f}")
    print(f"  flat-top scale             R_FL   = {R_FL:.4f} mm")
    print(f"  flat-top edge order        q      = {Q_ORDER}")
    print()
    print(f"  --> OUTPUT flat-top radius        = {Rmax:.4f} mm  (dia {2*Rmax:.3f} mm)")
    print(f"  --> areal magnification           = {(Rmax/R_TRUNC)**2:.3f}")

    print()
    print("=" * 74)
    print("ZEMAX LENS DATA EDITOR  (Even Asphere, all coefficients in mm)")
    print("=" * 74)
    hdr = f"{'#':>2} {'Type':<14}{'Radius':>12}{'Thick':>10}{'Glass':>9}{'S-Diam':>9}"
    print(hdr)
    print("-" * 74)
    sd1 = R_TRUNC * APER_PAD
    sd2 = Rmax * APER_PAD
    print(f"{1:>2} {'Standard':<14}{'Infinity':>12}{CT1:>10.3f}{'SILICA':>9}{sd1:>9.3f}")
    print(f"{2:>2} {'Even Asphere':<14}{1/c1:>12.4f}{D_GAP:>10.3f}{'':>9}{sd1:>9.3f}")
    print(f"{3:>2} {'Even Asphere':<14}{1/c2:>12.4f}{CT2:>10.3f}{'SILICA':>9}{sd2:>9.3f}")
    print(f"{4:>2} {'Standard':<14}{'Infinity':>12}{'--':>10}{'':>9}{sd2:>9.3f}")
    print()
    for lbl, kk, aa, rr in (("Surface 2", k1, a1, rms1), ("Surface 3", k2, a2, rms2)):
        print(f"{lbl}:  conic k = {kk: .6f}")
        for i, ai in enumerate(aa):
            print(f"            {2*i+4:>2}th order term = {ai: .6e}")
        print(f"            sag fit RMS  = {rr*1e6:.3f} nm  ({rr/(LAMBDA_UM*1e-3):.4f} waves)")
        print()

    print("=" * 74)
    print("VERIFICATION: real-ray trace through the FITTED surfaces")
    print("=" * 74)
    rs = np.linspace(1e-4, R_TRUNC, 400)
    Rt, ang = [], []
    for ri in rs:
        out = trace(ri, c1, k1, a1, c2, k2, a2)
        Rt.append(out[0])
        ang.append(np.degrees(np.arctan2(out[1][0], out[1][1])))
    Rt = np.array(Rt); ang = np.array(ang)

    R_ideal = ray_map(rs)
    print(f"  max |R_traced - R_ideal|     = {np.abs(Rt-R_ideal).max()*1000:.3f} um")
    print(f"  max output ray angle          = {np.abs(ang).max()*1000:.4f} mdeg"
          f"   ({np.abs(ang).max()*np.pi/180*1e6:.2f} urad)")

    # geometric irradiance from the traced map: I_out(R) = I_in(r) * (r/R) * dr/dR
    Iin = np.exp(-2.0 * (rs / W0) ** 2)
    drdR = np.gradient(rs, Rt)
    Iout = Iin * (rs / Rt) * drdR
    core = Rt < 0.90 * Rt.max()
    E = Iout[core]
    gamma = 1.0 - np.sqrt(np.mean((E - E.mean()) ** 2)) / E.mean()
    print(f"  geometric uniformity gamma    = {100*gamma:.2f} %  (paper's eq. 13,"
          f" inner 90% of the flat top)")
    print(f"  peak-to-valley in that region = {100*(E.max()-E.min())/E.mean():.2f} %")
    print()

    # sag tables -- this is what an asphere vendor actually wants, not coefficients
    np.savetxt("surface2_sag.csv", np.column_stack([r_fit, z1_fit]),
               delimiter=",", header="r_mm,sag_mm", comments="")
    np.savetxt("surface3_sag.csv", np.column_stack([R_fit, z2_fit]),
               delimiter=",", header="r_mm,sag_mm", comments="")
    print("  wrote surface2_sag.csv / surface3_sag.csv (exact sag, for the vendor)")
    print()

    print("=" * 74)
    print("BEAM EXPANDER (stage 2)")
    print("=" * 74)
    for target in (30.0, 40.0, 50.0):
        M = target / (2 * Rmax)
        print(f"  {2*Rmax:.2f} mm -> {target:.0f} mm  requires afocal M = {M:.3f}"
              f"   (e.g. f1 = -50 mm, f2 = {50*M:.1f} mm, spacing {50*M-50:.1f} mm)")
    print()
    print(f"  diffraction-limited full divergence of a {2*Rmax*3:.0f} mm top hat:"
          f" ~{2*1.22*LAMBDA_UM*1e-3/(2*Rmax*3)*1e6:.1f} urad")
    print("  the paper measured 0.6 deg = 10 mrad, i.e. ~1000x that -- residual")
    print("  collimation error, not a fundamental limit.")


DESIGN INPUTS
  wavelength                 638.0 nm
  input Gaussian 1/e^2 waist w0     = 5.5000 mm  (dia 11.000 mm)
  input truncation radius    r_max  = 5.5000 mm
  truncation ratio           r_max/w0 = 1.000
  power inside r_max         = 86.47 %
  glass index                n      = 1.45680
  asphere vertex separation  D      = 30.000 mm
  on-axis magnification      m0     = 2.3227
  flat-top scale             R_FL   = 9.0332 mm
  flat-top edge order        q      = inf

  --> OUTPUT flat-top radius        = 8.3997 mm  (dia 16.799 mm)
  --> areal magnification           = 2.332

ZEMAX LENS DATA EDITOR  (Even Asphere, all coefficients in mm)
 # Type                Radius     Thick    Glass   S-Diam
--------------------------------------------------------------------------
 1 Standard          Infinity     5.000   SILICA    5.830
 2 Even Asphere       10.3606    30.000             5.830
 3 Even Asphere       24.0646     5.000   SILICA    8.904
 4 Standard          Infinity        -- 